## Section 1 - Training Functions

These functions handle learning the BPE vocabulary from a piece of text: 
- counting word frequencies
- representing words as symbols
- counting pair frequencies
- iteratively merging the most common pairs

In [1]:
def get_word_freqs(text):
    """
    Split text into words and count how often each word appears.
    Returns a dict like {'low': 3, 'lower': 1, 'lowest': 1}
    """
    words = text.split()
    word_freqs = {}
    for word in words:
        word_freqs[word] = word_freqs.get(word, 0) + 1
    return word_freqs

In [2]:
def words_to_symbols(word_freqs):
    """
    Convert each word into a list of characters + end-of-word marker.
    Returns a dict like {('l','o','w','_'): 3, ('l','o','w','e','r','_'): 1, ...}
    We use a tuple of symbols as the key (tuples are hashable, lists aren't).
    """
    symbol_freqs = {}
    for word, freq in word_freqs.items():
        symbols = tuple(list(word) + ['_'])
        symbol_freqs[symbols] = freq
    return symbol_freqs

In [3]:
def get_pair_freqs(symbol_freqs):
    """
    Count how often each adjacent pair of symbols appears,
    weighted by word frequency.
    Returns a dict like {('l','o'): 5, ('o','w'): 5, ...}
    """
    pair_freqs = {}
    for symbols, freq in symbol_freqs.items():
        for i in range(len(symbols) - 1):
            pair = (symbols[i], symbols[i + 1])
            pair_freqs[pair] = pair_freqs.get(pair, 0) + freq
    return pair_freqs

In [4]:
def get_best_pair(pair_freqs):
    """
    Find the pair with the highest frequency.
    Returns None if there are no pairs left.
    """
    if not pair_freqs:
        return None
    best_pair = max(pair_freqs, key=pair_freqs.get)
    return best_pair

In [5]:
def merge_pair(symbol_freqs, pair_to_merge):
    """
    Replace every occurrence of pair_to_merge with a single merged symbol,
    across all words in symbol_freqs.
    """
    new_symbol_freqs = {}
    merged_symbol = pair_to_merge[0] + pair_to_merge[1]  # e.g. 'l' + 'o' -> 'lo'

    for symbols, freq in symbol_freqs.items():
        new_symbols = []
        i = 0
        while i < len(symbols):
            # Check if the pair starting at position i matches what we're merging
            if i < len(symbols) - 1 and (symbols[i], symbols[i + 1]) == pair_to_merge:
                new_symbols.append(merged_symbol)
                i += 2   # skip both symbols we just merged
            else:
                new_symbols.append(symbols[i])
                i += 1   # just move forward normally

        new_symbol_freqs[tuple(new_symbols)] = freq

    return new_symbol_freqs

In [6]:
def train_bpe(text, target_vocab_size):
    """
    Train a BPE tokenizer on the given text.
    Returns:
        merges: list of merge rules in the order they were learned
        vocab: set of all symbols in the final vocabulary
    """
    word_freqs = get_word_freqs(text)
    symbol_freqs = words_to_symbols(word_freqs)

    # Start vocab with every individual character we've seen, plus the end marker
    vocab = set()
    for symbols in symbol_freqs:
        vocab.update(symbols)

    merges = []  # we'll record merge rules in order here

    while len(vocab) < target_vocab_size:
        pair_freqs = get_pair_freqs(symbol_freqs)
        best_pair = get_best_pair(pair_freqs)

        if best_pair is None:
            print("No more pairs to merge — stopping early.")
            break

        symbol_freqs = merge_pair(symbol_freqs, best_pair)
        merges.append(best_pair)

        merged_symbol = best_pair[0] + best_pair[1]
        vocab.add(merged_symbol)

    return merges, vocab

## Section 2 - Encoding/decoding functions

These functions handle the application of the learned BPE vocabulary:
- applying a single merge rule
- encoding a single word based on all rules
- decoding the symbols back into a single word
- encoding an entire text
- decoding a series of encoded text

In [7]:
def apply_merge(symbols, pair_to_merge):
    """
    Apply a single merge rule to one word's tuple of symbols.
    Returns a new tuple with the pair merged wherever it appears.
    """
    merged_symbol = pair_to_merge[0] + pair_to_merge[1]
    new_symbols = []
    i = 0
    while i < len(symbols):
        if i < len(symbols) - 1 and (symbols[i], symbols[i + 1]) == pair_to_merge:
            new_symbols.append(merged_symbol)
            i += 2
        else:
            new_symbols.append(symbols[i])
            i += 1
    return tuple(new_symbols)

In [8]:
def encode_word(word, merges):
    """
    Encode a single word using the learned merge rules, in order.
    """
    symbols = tuple(list(word) + ['_'])
    for pair in merges:
        symbols = apply_merge(symbols, pair)
    return symbols

In [9]:
def decode(symbols):
    """
    Convert a tuple of symbols back into a readable string.
    """
    text = "".join(symbols)
    text = text.replace("_", " ")
    return text.strip()

In [10]:
def encode_text(text, merges):
    """
    Encode a full piece of text (multiple words) using learned merge rules.
    Returns a list of symbol-tuples, one per word.
    """
    words = text.split()
    return [encode_word(word, merges) for word in words]

In [11]:
def decode_text(encoded_words):
    """
    Decode a list of symbol-tuples (one per word) back into a sentence.
    """
    words = [decode(symbols) for symbols in encoded_words]
    return " ".join(words)

## Section 3 - Demonstrating and testing

Before applying this tokenizer to real text, we verify correctness on a small, fully traceable example: a 5-word toy vocabulary ("low", "low", "low", "lower", "lowest"). Every merge step below was predicted by hand before running the code. The encode/decode tests include words that were not seen during training to confirm the tokenizer generalizes well when it encounters something unfamiliar.

Toy training data and full hand-traced walkthrough:

In [12]:
text = "low low low lower lowest"
word_freqs = get_word_freqs(text)
print(word_freqs)

symbol_freqs = words_to_symbols(word_freqs)
print(symbol_freqs)

{'low': 3, 'lower': 1, 'lowest': 1}
{('l', 'o', 'w', '_'): 3, ('l', 'o', 'w', 'e', 'r', '_'): 1, ('l', 'o', 'w', 'e', 's', 't', '_'): 1}


In [13]:
pair_freqs = get_pair_freqs(symbol_freqs)
print(pair_freqs)

{('l', 'o'): 5, ('o', 'w'): 5, ('w', '_'): 3, ('w', 'e'): 2, ('e', 'r'): 1, ('r', '_'): 1, ('e', 's'): 1, ('s', 't'): 1, ('t', '_'): 1}


In [14]:
best_pair = get_best_pair(pair_freqs)
print(best_pair)

('l', 'o')


In [15]:
new_symbol_freqs = merge_pair(symbol_freqs, best_pair)
print(new_symbol_freqs)

{('lo', 'w', '_'): 3, ('lo', 'w', 'e', 'r', '_'): 1, ('lo', 'w', 'e', 's', 't', '_'): 1}


Full training run on the toy example:

In [16]:
text = "low low low lower lowest"
merges, vocab = train_bpe(text, target_vocab_size=20)

print("Merges learned, in order:")
for m in merges:
    print(m)

print("\nFinal vocab:")
print(vocab)

No more pairs to merge — stopping early.
Merges learned, in order:
('l', 'o')
('lo', 'w')
('low', '_')
('low', 'e')
('lowe', 'r')
('lower', '_')
('lowe', 's')
('lowes', 't')
('lowest', '_')

Final vocab:
{'lower_', 'l', 'low_', 'lower', 'lowest', 's', 'lowest_', 'lo', 'lowe', '_', 'lowes', 'r', 'w', 't', 'low', 'o', 'e'}


Testing encode on known and unseen words:

In [17]:
print(encode_word("slow", merges))
print(encode_word("loan", merges))
print(encode_word("lime", merges))
print(encode_word("slower", merges))

('s', 'low_')
('lo', 'a', 'n', '_')
('l', 'i', 'm', 'e', '_')
('s', 'lower_')


Testing decode (round trip verification):

In [18]:
print(decode(('s', 'low_')))
print(decode(('lo', 'a', 'n', '_')))
print(decode(('l', 'i', 'm', 'e', '_')))
print(decode(('s', 'lower_')))

slow
loan
lime
slower


Testing full sentence encode/decode:

In [19]:
test_text = "low lower lime"
encoded = encode_text(test_text, merges)
print(encoded)

decoded = decode_text(encoded)
print(decoded)

[('low_',), ('lower_',), ('l', 'i', 'm', 'e', '_')]
low lower lime


### What this confirmed

The hand-traced predictions matched the code's output exactly, including an instructive edge case: because this toy vocabulary only contains 3 unique words, the algorithm eventually runs out of meaningful frequency signal and starts merging pairs that only occur once. "lower" and "lowest" end up fully merged into single whole-word tokens - not because they are common English words, but because by that point only tied pairs at frequency = 1 remained, and the algorithm had nothing left to meaningfully compare them against. This is a useful early warning sign for what we would expect to see more clearly at larger scale: a tokenizer's vocabulary is entirely a reflection of its training data's statistics, not of any linguistic notion of what "deserves" to be a token.

The encode test on "slower" (a word combining a known prefix with an unseen full word) and "lime" (sharing no merge patterns with training data at all) confirms the tokenizer generalizes well, falling back to individual characters rather than failing when it meets something outside its training vocabulary.

## Section 4 - Real-world Application - English Prose

Training on a 5-word toy example confirms the algorithm works correctly, but real text reveals patterns - and limitations - that don't appear at small scale. This section trains the tokenizer on the full text of *Alice's Adventures in Wonderland*, a public domain book from Project Gutenberg, to see what a tokenizer trained on real English prose actually learns.

Opening and storing the text:

In [20]:
with open("alice.txt", "r", encoding="utf-8") as f:
    alice_text = f.read()

print(len(alice_text))       # total characters
print(alice_text[:300])      # peek at the beginning

144696
*** START OF THE PROJECT GUTENBERG EBOOK 11 ***

[Illustration]




Alice’s Adventures in Wonderland

by Lewis Carroll

THE MILLENNIUM FULCRUM EDITION 3.0

Contents

 CHAPTER I.     Down the Rabbit-Hole
 CHAPTER II.    The Pool of Tears
 CHAPTER III.   A Caucus-Race and a Long Tale
 CHAPTER IV.    T


Using start and end markers to extract the clean text content in between: 

The actual header is "*** START OF THE PROJECT GUTENBERG EBOOK 11 ***". 

Due to other Project Gutenberg Ebooks having different identification numbers we search for the trailing "***" to determine where to start the slice.

In [21]:
start_marker = "*** START OF THE PROJECT GUTENBERG EBOOK"
end_marker = "*** END OF THE PROJECT GUTENBERG EBOOK"
marker_search_start = alice_text.find(start_marker)
header_end = alice_text.find("***", marker_search_start + len(start_marker))
content_start = header_end + 3
end_index = alice_text.find(end_marker)

clean_text = alice_text[content_start : end_index]
print(len(clean_text))
print(clean_text[:200])

144603


[Illustration]




Alice’s Adventures in Wonderland

by Lewis Carroll

THE MILLENNIUM FULCRUM EDITION 3.0

Contents

 CHAPTER I.     Down the Rabbit-Hole
 CHAPTER II.    The Pool of Tears
 CHAPTER I


Training the BPE with the extracted English prose text:

In [22]:
merges, vocab = train_bpe(clean_text, target_vocab_size=500)
print(f"Number of merges learned: {len(merges)}")
print(f"Final vocab size: {len(vocab)}")

Number of merges learned: 427
Final vocab size: 500


In [23]:
print("First 20 merges learned:")
for m in merges[:20]:
    print(m)

print("\nLast 20 merges learned:")
for m in merges[-20:]:
    print(m)

First 20 merges learned:
('e', '_')
('t', 'h')
('d', '_')
('t', '_')
('i', 'n')
(',', '_')
('e', 'r')
('s', '_')
('o', 'u')
('th', 'e_')
('a', 'n')
('o', '_')
('”', '_')
('y', '_')
('o', 'n')
('in', 'g')
('e', 'n')
('an', 'd_')
('s', 'h')
('t', 'o_')

Last 20 merges learned:
('“', 'H')
('k', ',_')
('sa', 'y_')
('af', 'ter_')
('l', 'ing_')
('gh', '_')
('p', 'er')
('h', ',_')
('an', 'g')
('be', 'en_')
('l', 'ed_')
('t', 'on')
('rou', 'nd_')
('c', 'an')
('es', 's_')
('t', 'er')
('c', 'ur')
('i', 's,_')
('S', 'he_')
('so', 'me')


### What this revealed

The early merges are dominated by extremely common English patterns, including not only letter pairs like `('t','h')`, but also word-boundary patterns like `('e','_')`, reflecting how many English words end in "e". Notably, "the" (the most common word in English) assembles itself piece by piece purely from frequency: `('t','h')` merges early, `('e','_')` merges separately, and eventually `('th','e_')` combines them into a single token representing the whole word "the ".

This also surfaces a real limitation worth naming directly: because word splitting here is based only on whitespace, punctuation stays attached to words. Merges like `('i','s,_')` treat "is," as an entirely different token from "is", despite being grammatically the same word. This is a deliberate finding, not an oversight. It is documented as a known limitation and addressed as a "Version 2" improvement (see version2_goals.md), which separates punctuation from words before BPE training begins.

## Section 5 - Comparison Case - Python Code

Code and natural language prose have different statistical structure: code has more unique, project-specific identifiers (variable names chosen by an author) and a narrower set of highly repeated keywords and syntax characters. This section tests that hypothesis directly by training the same tokenizer on a small Python code sample - three of the functions defined in Section 1 - using a smaller target vocabulary size, since the sample itself is much shorter than a full novel.

Example text taken from this project's Python code:

In [24]:
python_code = '''
def get_word_freqs(text):
    words = text.split()
    word_freqs = {}
    for word in words:
        word_freqs[word] = word_freqs.get(word, 0) + 1
    return word_freqs


def words_to_symbols(word_freqs):
    symbol_freqs = {}
    for word, freq in word_freqs.items():
        symbols = tuple(list(word) + ['_'])
        symbol_freqs[symbols] = freq
    return symbol_freqs


def get_pair_freqs(symbol_freqs):
    pair_freqs = {}
    for symbols, freq in symbol_freqs.items():
        for i in range(len(symbols) - 1):
            pair = (symbols[i], symbols[i + 1])
            pair_freqs[pair] = pair_freqs.get(pair, 0) + freq
    return pair_freqs
'''

Observing the first 20 split words for examples and identifying patterns:

In [25]:
words = python_code.split()
print(words[:20])

['def', 'get_word_freqs(text):', 'words', '=', 'text.split()', 'word_freqs', '=', '{}', 'for', 'word', 'in', 'words:', 'word_freqs[word]', '=', 'word_freqs.get(word,', '0)', '+', '1', 'return', 'word_freqs']


Training the BPE with the Python code example:

In [26]:
code_merges, code_vocab = train_bpe(python_code, target_vocab_size=150)
print(f"Number of merges learned: {len(code_merges)}")
print(f"Final vocab size: {len(code_vocab)}")

print("\nFirst 15 merges:")
for m in code_merges[:15]:
    print(m)

print("\nLast 15 merges:")
for m in code_merges[-15:]:
    print(m)

Number of merges learned: 114
Final vocab size: 150

First 15 merges:
('r', 'e')
('f', 're')
('fre', 'q')
('o', 'r')
('_', 'freq')
('_freq', 's')
('w', 'or')
('wor', 'd')
('s', 'y')
('sy', 'm')
('sym', 'b')
('symb', 'o')
('symbo', 'l')
('=', '_')
('p', 'a')

Last 15 merges:
('tuple(lis', 't(')
('tuple(list(', 'word')
('tuple(list(word', ')_')
('[', "'")
("['", '_')
("['_", "'")
("['_'", '])_')
('symbol_freqs', '[')
('symbol_freqs[', 'symbols')
('symbol_freqs[symbols', ']_')
('get_', 'pair_freqs')
('get_pair_freqs', '(')
('get_pair_freqs(', 'symbol_freqs')
('get_pair_freqs(symbol_freqs', '):_')
('symbols', ',_')


### What this revealed

The comparison confirms the hypothesis. The earliest merges quickly assemble recognizable fragments (`freq`, `word`, and `symbol`), reflecting how often `symbol_freqs`, `word_freqs`, and `pair_freqs` repeat throughout this short sample. If we had used a wide variety of Python code, by many different users, on many different applications, perhaps more common Python commands and syntax would have a higher frequency. 

The algorithm runs out of genuine pattern diversity well before reaching the target vocabulary size. By the final few merges, the tokenizer is reconstructing entire fragments of source code character-by-character: the literal sequence `['_'])` (drawn directly from this project's own `words_to_symbols` function) is rebuilt across four consecutive merges near the end of training. 

This is the same "running out of meaningful frequency signal" behavior observed in Section 3 with "lower" and "lowest". It illustrates why training data size and diversity matter relative to target vocabulary size: a tokenizer trained on a handful of similar files would overfit to that exact code, producing a vocabulary unlikely to generalize well to other Python code with different naming conventions and structure.

The same "Version 2" improvement noted in Section 4 of separating punctuation and special characters before BPE training may matter even more for code than for prose. Some coders may mitigate this with white space for readability, but not all. Code contains many more structurally meaningful symbols (`(`, `)`, `[`, `]`, `:`, `,`, `=`) than English text does, and the runaway merging seen here suggests that without separating them out first, a code-trained tokenizer risks memorizing exact syntax patterns rather than learning generalizable subword units.